# Fase 3: Desenvolvimento do Modelo (O Cérebro)
Neste notebook, implementamos o pipeline completo para:
1. **Reproduzir a Baseline:** Validar o modelo base (EfficientNet-B7) usando os pesos fornecidos no repositório (`efficientnet_b7.pth`) e obter a métrica de referência (Macro F1-Score de ~0.738).
2. **Seleção de Modelos:** Treinar e comparar diferentes arquiteturas de CNNs e Transformers (DenseNet121, ConvNeXt, Swin Transformers, etc.) utilizando a biblioteca `timm`.
3. **Fine-Tuning & Otimização:** Ajustar hiperparâmetros, aplicar scheduler e tratar o desequilíbrio de classes usando pesos na função de perda.
4. **Experiment Tracking:** Utilizar o Weights & Biases (`wandb`) para monitorizar perdas, F1-scores, curvas de aprendizagem e matrizes de confusão.

In [7]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset
import torchvision.models as models
import timm

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from torchmetrics.classification import MulticlassF1Score

# MONAI Imports
from monai.transforms import (
    LoadImage, EnsureChannelFirst, Resize, NormalizeIntensity, ToTensor, Compose, Lambda,
    RandRotate, RandZoom, RandAdjustContrast, RandGaussianNoise, Activations, AsDiscrete
)
from monai.data import decollate_batch, DataLoader
from monai.metrics import ROCAUCMetric
from monai.utils import set_determinism
from monai.config import print_config

# Experiment Tracking
import wandb

print("Bibliotecas importadas com sucesso!")
print_config()

Bibliotecas importadas com sucesso!
MONAI version: 1.5.2
Numpy version: 2.4.6
Pytorch version: 2.12.0+cpu
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: d18565fb3e4fd8c556707f91ac280a2dc3f681c1
MONAI __file__: c:\Users\<username>\Documents\imagiologia\.venv\Lib\site-packages\monai\__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: NOT INSTALLED or UNKNOWN VERSION.
scikit-image version: NOT INSTALLED or UNKNOWN VERSION.
scipy version: 1.17.1
Pillow version: 12.2.0
Tensorboard version: NOT INSTALLED or UNKNOWN VERSION.
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.27.0+cpu
tqdm version: 4.67.3
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.2.2
pandas version: 3.0.3
einops version: NOT INSTALLED or UNKNOWN VERSION.
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED

In [8]:
# Configurar reprodutibilidade (Seed fixada)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_determinism(seed=seed)

set_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Treino configurado para utilizar dispositivo: {device}")

Treino configurado para utilizar dispositivo: cpu


### 1. Autenticação e Configuração do WandB
Insira a sua chave de API para fazer login no Weights & Biases.

In [9]:
# Descomente a linha abaixo para fazer login se necessário
# wandb.login()
print("WandB pronto para inicializar experiências!")

WandB pronto para inicializar experiências!


### 2. Carregamento dos Splits do Dataset (Train / Val / Test)
Definimos o caminho relativo do dataset apontando para os dados fixados e divididos por classes no repositório da baseline.

In [10]:
# Caminho relativo para a pasta do dataset fixo na baseline
base_dir = '../baseline_repo/MIQR-CC-Dataset/training/dataset'
phases = ['train', 'val', 'test']

data = {phase: {'images': [], 'labels': []} for phase in phases}
class_names = sorted([
    x for x in os.listdir(os.path.join(base_dir, 'train')) 
    if os.path.isdir(os.path.join(base_dir, 'train', x))
])
num_class = len(class_names)

def load_images_labels(phase):
    for i, class_name in enumerate(class_names):
        class_dir = os.path.join(base_dir, phase, class_name)
        valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
        image_files = [
            os.path.join(class_dir, x)
            for x in os.listdir(class_dir)
            if x.lower().endswith(valid_exts)
        ]
        data[phase]['images'].extend(image_files)
        data[phase]['labels'].extend([i] * len(image_files))

for phase in phases:
    load_images_labels(phase)

for phase in phases:
    print(f"Instâncias em {phase.capitalize()}: {len(data[phase]['images'])}")
print("Classes identificadas:", class_names)

Instâncias em Train: 1067
Instâncias em Val: 234
Instâncias em Test: 267
Classes identificadas: ['Biliary_Leaks', 'Lithiasis', 'Normal', 'Stricture']


### 3. Data Augmentation e Preprocessing (Transforms)
Definimos as transformações do MONAI para pré-processamento das imagens, garantindo que as imagens de 1 canal sejam replicadas para 3 canais (RGB) de modo a serem compatíveis com modelos pré-treinados em ImageNet.

In [ ]:
def repeat_channels(img):
    if img.shape[0] == 1:
        return img.repeat(3, 1, 1)
    return img

# Transforms para Treino (com Aumentação)
train_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((224, 224)),
    RandRotate(range_x=15, prob=0.5),
    RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.5),
    RandAdjustContrast(prob=0.5),
    RandGaussianNoise(prob=0.3, mean=0.0, std=0.01),
    NormalizeIntensity(),
    Lambda(repeat_channels),
    ToTensor(),
    ])

# Transforms para Validação e Teste (Sem Aumentações, apenas pré-processamento)
val_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((224, 224)),
    NormalizeIntensity(),
    Lambda(repeat_channels),
    ToTensor(),
    ])

print("Transforms configurados com sucesso!")

Transforms configurados com sucesso!


### 4. Criação do Dataset Customizado e Dataloaders

In [12]:
class ERCPDataset(Dataset):
    def __init__(self, image_files, labels, transforms):
        self.image_files = image_files
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        image_path = self.image_files[index]
        return self.transforms(image_path), self.labels[index]

train_ds = ERCPDataset(data['train']['images'], data['train']['labels'], train_transforms)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)

val_ds = ERCPDataset(data['val']['images'], data['val']['labels'], val_transforms)
val_loader = DataLoader(val_ds, batch_size=8, num_workers=0)

test_ds = ERCPDataset(data['test']['images'], data['test']['labels'], val_transforms)
test_loader = DataLoader(test_ds, batch_size=8, num_workers=0)

print(f"DataLoaders prontos. Tamanho do batch de treino: {train_loader.batch_size}")

DataLoaders prontos. Tamanho do batch de treino: 8


### 5. Função de Instanciação de Modelos SOTA (`torchvision` & `timm`)
Suporta carregar modelos clássicos do torchvision e os modelos do timm.

In [13]:
def create_target_model(model_name, num_classes=4):
    if model_name == 'efficientnet_b7':
        model = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'densenet121':
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        # Carregar do timm (ViTs, ConvNeXt, Swin, etc.)
        print(f"Carregando model '{model_name}' do timm...")
        model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    return model.to(device)

### 6. Configuração de Pesos das Classes para Combater Desequilíbrio
Aplicamos pesos inversamente proporcionais à frequência das classes para mitigar o desequilíbrio no treino.

In [14]:
# Frequência de amostras por classe no set de treino:
# Biliary_Leaks: 110, Lithiasis: 505, Normal: 197, Stricture: 255
class_counts = [110, 505, 197, 255]
total_samples = sum(class_counts)

# Frequência inversa
class_weights = [total_samples / (len(class_counts) * c) for c in class_counts]
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Pesos calculados para as classes:", class_weights)

loss_function = nn.CrossEntropyLoss(weight=class_weights_tensor)

Pesos calculados para as classes: [2.425, 0.5282178217821782, 1.3540609137055837, 1.046078431372549]


### 7. Loop de Treinamento Geral com Tracking do WandB
O loop monitoriza o macro F1-score e o AUC, implementa o decaimento do learning rate via Cosine Annealing, faz early stopping se a validação parar de melhorar e envia as métricas e a matriz de confusão final para o WandB.

In [15]:
def train_model(model_name, train_dl, val_dl, epochs=25, lr=1e-5):
    # Inicializar o WandB Run
    run = wandb.init(
        project="ERCP-Imagiologia-Phase3",
        config={
            "model": model_name,
            "epochs": epochs,
            "learning_rate": lr,
            "batch_size": train_dl.batch_size,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "early_stopping_patience": 10
        }
    )
    config = wandb.config

    model = create_target_model(model_name)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    
    # Métricas
    f1_metric = MulticlassF1Score(num_classes=num_class, average='macro').to(device)
    auc_metric = ROCAUCMetric()
    softmax_act = Activations(softmax=True)
    to_onehot = AsDiscrete(to_onehot=num_class)

    best_metric = -1
    best_metric_epoch = -1
    epochs_without_improvement = 0

    os.makedirs('./models', exist_ok=True)
    model_save_path = f"./models/{model_name}_best.pth"

    for epoch in range(config.epochs):
        model.train()
        running_loss = 0.0
        running_corrects = 0.0
        y_train_all = torch.tensor([], dtype=torch.long, device=device)
        y_train_pred = torch.tensor([], dtype=torch.float32, device=device)

        for inputs, labels in train_dl:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.detach() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)

            y_train_pred = torch.cat([y_train_pred, outputs], dim=0)
            y_train_all = torch.cat([y_train_all, labels], dim=0)

        epoch_loss = running_loss / len(train_dl.dataset)
        epoch_acc = running_corrects.float() / len(train_dl.dataset)
        epoch_f1 = f1_metric(y_train_pred.argmax(dim=1), y_train_all).item()

        # Validação
        model.eval()
        val_loss = 0.0
        val_corrects = 0.0
        y_val_all = torch.tensor([], dtype=torch.long, device=device)
        y_val_pred = torch.tensor([], dtype=torch.float32, device=device)

        with torch.no_grad():
            for val_images, val_labels in val_dl:
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                outputs = model(val_images)
                loss = loss_function(outputs, val_labels)
                
                val_loss += loss.detach() * val_images.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == val_labels.data)

                y_val_pred = torch.cat([y_val_pred, outputs], dim=0)
                y_val_all = torch.cat([y_val_all, val_labels], dim=0)

            epoch_val_loss = val_loss / len(val_dl.dataset)
            epoch_val_acc = val_corrects.float() / len(val_dl.dataset)
            epoch_val_f1 = f1_metric(y_val_pred.argmax(dim=1), y_val_all).item()

            # Calcular AUC usando MONAI metrics
            y_onehot = [to_onehot(i) for i in decollate_batch(y_val_all, detach=False)]
            y_pred_act = [softmax_act(i) for i in decollate_batch(y_val_pred)]
            y_pred_act = torch.cat(y_pred_act, dim=0)
            y_onehot = torch.cat(y_onehot, dim=0)
            auc_metric(y_pred_act, y_onehot)
            epoch_val_auc = auc_metric.aggregate()
            auc_metric.reset()

            # Logging de métricas no WandB
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": epoch_loss.item(),
                "train_accuracy": epoch_acc.item(),
                "train_f1_macro": epoch_f1,
                "val_loss": epoch_val_loss.item(),
                "val_accuracy": epoch_val_acc.item(),
                "val_f1_macro": epoch_val_f1,
                "val_auc": epoch_val_auc,
                "learning_rate": optimizer.param_groups[0]['lr']
            })

            print(f"Epoch {epoch+1:02d} | Train Loss: {epoch_loss:.4f} F1: {epoch_f1:.4f} | Val Loss: {epoch_val_loss:.4f} F1: {epoch_val_f1:.4f} AUC: {epoch_val_auc:.4f}")

            # Guardar o melhor modelo (baseado em Val F1 Macro)
            if epoch_val_f1 > best_metric:
                best_metric = epoch_val_f1
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), model_save_path)
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            # Early Stopping
            if epochs_without_improvement >= config.early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}. Best Val F1: {best_metric:.4f} at epoch {best_metric_epoch}.")
                break

        scheduler.step()

    print(f"Treino terminado para {model_name}. Melhor F1 Macro: {best_metric:.4f} na época {best_metric_epoch}.")
    
    # Carregar melhor modelo para a avaliação de teste
    model.load_state_dict(torch.load(model_save_path))
    evaluate_and_log_test(model, test_loader, model_name, f1_metric, run)
    run.finish()

def evaluate_and_log_test(model, test_dl, model_name, f1_metric, run):
    model.eval()
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for images, labels in test_dl:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            
            y_true_all.extend(labels.numpy())
            y_pred_all.extend(preds.cpu().numpy())
            
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)

    test_f1 = f1_score(y_true_all, y_pred_all, average='macro')
    print(f"Test set F1 Score (Macro) para {model_name}: {test_f1:.4f}")
    print(classification_report(y_true_all, y_pred_all, target_names=class_names, digits=4, zero_division=0))

    # Log final de métricas de teste no WandB
    wandb.log({"test_f1_macro": test_f1})

    # Log da Matriz de Confusão no WandB
    wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true_all,
        preds=y_pred_all,
        class_names=class_names
    )})

### 8. Validação e Reprodução da Baseline (EfficientNet-B7)
Nesta secção, carregamos os pesos salvos da baseline e rodamos a avaliação no Test Loader para validar se o Macro F1-score bate com a referência de 0.7381.

In [16]:
# Instanciar a baseline
baseline_model = create_target_model('efficientnet_b7')

# Carregar pesos salvos da baseline
baseline_ckpt = '../baseline_repo/MIQR-CC-Dataset/training/models/efficientnet_b7.pth'
baseline_model.load_state_dict(torch.load(baseline_ckpt, map_location=device))
baseline_model.eval()

print("Pesos da baseline carregados com sucesso!")

# Avaliação
y_true_all = []
y_pred_all = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = baseline_model(images)
        preds = outputs.argmax(dim=1)
        y_true_all.extend(labels.numpy())
        y_pred_all.extend(preds.cpu().numpy())

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

baseline_f1 = f1_score(y_true_all, y_pred_all, average='macro')
print(f"[VALIDAÇÃO] Macro F1-Score do Modelo Base (EfficientNet-B7) no Test Set: {baseline_f1:.4f}")
print(classification_report(y_true_all, y_pred_all, target_names=class_names, digits=4, zero_division=0))

Downloading: "https://download.pytorch.org/models/efficientnet_b7_lukemelas-c5b4e57e.pth" to C:\Users\Smartglobe/.cache\torch\hub\checkpoints\efficientnet_b7_lukemelas-c5b4e57e.pth


100%|██████████| 255M/255M [01:31<00:00, 2.91MB/s] 


Pesos da baseline carregados com sucesso!
[VALIDAÇÃO] Macro F1-Score do Modelo Base (EfficientNet-B7) no Test Set: 0.7381
               precision    recall  f1-score   support

Biliary_Leaks     0.6923    0.5294    0.6000        17
    Lithiasis     0.8532    0.7561    0.8017       123
       Normal     0.5714    0.9302    0.7080        43
    Stricture     0.8933    0.7976    0.8428        84

     accuracy                         0.7828       267
    macro avg     0.7526    0.7533    0.7381       267
 weighted avg     0.8102    0.7828    0.7867       267



### 9. Treino e Otimização de Outros Modelos Comparativos
Descomente e execute as linhas abaixo para treinar e monitorizar diferentes arquiteturas no WandB.

In [ ]:
# Exemplo de treino da DenseNet121
train_model('densenet121', train_loader, val_loader, epochs=25, lr=2e-5)

# Exemplo de treino de uma Vision Transformer (DeiT III) via timm
train_model('deit3_base_patch16_384', train_loader, val_loader, epochs=20, lr=1e-5)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter you

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\Smartglobe/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:12<00:00, 2.67MB/s]


Epoch 01 | Train Loss: 1.3565 F1: 0.2961 | Val Loss: 1.3392 F1: 0.2581 AUC: 0.6079
Epoch 02 | Train Loss: 1.2873 F1: 0.4252 | Val Loss: 1.3243 F1: 0.3089 AUC: 0.6398
Epoch 03 | Train Loss: 1.2094 F1: 0.4866 | Val Loss: 1.3007 F1: 0.3739 AUC: 0.6693
Epoch 04 | Train Loss: 1.1408 F1: 0.5371 | Val Loss: 1.2824 F1: 0.3368 AUC: 0.6755
Epoch 05 | Train Loss: 1.0581 F1: 0.5674 | Val Loss: 1.2241 F1: 0.3226 AUC: 0.7154
Epoch 06 | Train Loss: 0.9657 F1: 0.6204 | Val Loss: 1.2562 F1: 0.3791 AUC: 0.6971
Epoch 07 | Train Loss: 0.8806 F1: 0.6561 | Val Loss: 1.1652 F1: 0.4277 AUC: 0.7551
Epoch 08 | Train Loss: 0.8143 F1: 0.6892 | Val Loss: 1.1951 F1: 0.3822 AUC: 0.7393
Epoch 09 | Train Loss: 0.7697 F1: 0.7056 | Val Loss: 1.1372 F1: 0.4346 AUC: 0.7728
Epoch 10 | Train Loss: 0.6959 F1: 0.7278 | Val Loss: 1.1417 F1: 0.4740 AUC: 0.7725
Epoch 11 | Train Loss: 0.6686 F1: 0.7480 | Val Loss: 1.1223 F1: 0.4397 AUC: 0.7824
Epoch 12 | Train Loss: 0.6259 F1: 0.7522 | Val Loss: 1.1471 F1: 0.5334 AUC: 0.7748
Epoc